# Day 61 — Week 6 Mini-Project
**Month 3 | ShopEase 2024 Full-Year Analytics Pipeline**

---

> **Scenario:**
>
> The ShopEase leadership team wants a full-year 2024 performance report before the January board meeting.
> You have 600 orders across 4 regions, 5 categories, and 3 customer segments.
>
> Your job: build one end-to-end analytics pipeline that integrates everything from Week 6.
> Memory-efficient data loading → NumPy descriptive stats → time-series trend analysis →
> MultiIndex breakdowns → chunked production pipeline. No external code. One notebook. Boardroom ready.

---
**Score: 80 pts**  
Topics tested: Memory Optimization · NumPy · Time Series · MultiIndex & Advanced GroupBy · Chunked Pipeline


## Section 1 — Raw Data
*Run this cell first. Never modify it. Work on `df` throughout.*

In [1]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=61)
n   = 600

raw = pd.DataFrame({
    'order_id'    : [f'ORD{5000+i}' for i in range(n)],
    'customer_id' : rng.integers(1001, 1081, n).astype('int64'),
    'order_date'  : pd.to_datetime(
                        rng.choice(pd.date_range('2024-01-01','2024-12-31'), n, replace=True)
                    ),
    'region'      : rng.choice(['North','South','East','West'], n, p=[0.30,0.25,0.25,0.20]),
    'category'    : rng.choice(['Electronics','Clothing','Home','Sports','Books'], n,
                                p=[0.30,0.25,0.20,0.15,0.10]),
    'segment'     : rng.choice(['Regular','Premium','VIP'], n, p=[0.50,0.35,0.15]),
    'status'      : rng.choice(['Delivered','Returned','Pending'], n, p=[0.70,0.20,0.10]),
    'units'       : rng.integers(1, 25, n).astype('int64'),
    'unit_price'  : rng.uniform(50, 600, n).astype('float64'),
    'discount_pct': rng.choice([0,5,10,15,20], n).astype('int64'),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)

df = raw.copy()
print(f"Dataset: {df.shape[0]} orders | {df['order_date'].dt.year.unique()[0]} full year")
print(f"Regions: {sorted(df['region'].unique())} | Categories: {sorted(df['category'].unique())}")
print(f"Total revenue: ₹{df['revenue'].sum():,.2f}")
df.head(3)


Dataset: 600 orders | 2024 full year
Regions: ['East', 'North', 'South', 'West'] | Categories: ['Books', 'Clothing', 'Electronics', 'Home', 'Sports']
Total revenue: ₹2,249,284.13


,order_id,customer_id,order_date,region,category,segment,status,units,unit_price,discount_pct,revenue
0,ORD5000,1034,2024-04-04,West,Clothing,Regular,Delivered,4,225.157147,5,855.60
1,ORD5001,1030,2024-05-14,North,Home,Premium,Pending,2,186.330423,15,316.76
2,ORD5002,1061,2024-04-06,West,Books,Regular,Delivered,11,82.144213,0,903.59


## Task 1 — Memory Optimisation (12 pts)

**Context:** The production server runs this pipeline on 6 million rows. Before adding any analytics,
optimise the dtypes so the pipeline uses the minimum memory needed without losing precision on columns
you'll aggregate.

**Rule:** Do NOT downcast `revenue` — it will be summed across chunks in Task 5. Float32 accumulates rounding error.


**T1a (4 pts):** Compute the baseline total memory of `df` in KB using `memory_usage(deep=True)`.
Print each column's KB. Store total in `baseline_kb`.

In [2]:
# T1a — Baseline memory audit
# 1. df.memory_usage(deep=True) / 1024 → per-column KB
# 2. Print each column
# 3. baseline_kb = total
per_col_kb = df.memory_usage(deep=True) / 1024
print("Per-column memory (KB):")
print(per_col_kb.to_string())

baseline_kb = per_col_kb.sum()
print(f"\nTotal baseline memory: {baseline_kb:.2f} KB")



Per-column memory (KB):
Index            0.128906
order_id        37.500000
customer_id      4.687500
order_date       4.687500
region          36.052734
category        37.795898
segment         37.164062
status          38.458984
units            4.687500
unit_price       4.687500
discount_pct     4.687500
revenue          4.687500

Total baseline memory: 215.23 KB


**T1b (8 pts):** Create `df_opt = df.copy()`. Apply all safe downcasts:

| Column | → | Rule |
|--------|---|------|
| `customer_id` | int32 | max 1080 |
| `units` | int8 | max 24 |
| `discount_pct` | int8 | values 0,5,10,15,20 |
| `unit_price` | float32 | precision OK for display |
| `region`, `category`, `segment`, `status` | category | low cardinality |

**Do not downcast `revenue`.** Print optimised KB and % reduction. Write NRA insight.


In [3]:
# T1b — Dtype optimisation
# 1. df_opt = df.copy()
# 2. Apply all downcasts from the table (revenue stays float64)
# 3. optimised_kb = df_opt.memory_usage(deep=True).sum() / 1024
# 4. reduction = (1 - optimised_kb / baseline_kb) * 100
# 5. Print: "Optimised: XX.XX KB | Reduction: XX.X%"
# 6. NRA insight
df_opt = df.copy()
df_opt['customer_id']  = df_opt['customer_id'].astype('int32')
df_opt['units']        = df_opt['units'].astype('int8')
df_opt['discount_pct'] = df_opt['discount_pct'].astype('int8')
df_opt['unit_price']   = df_opt['unit_price'].astype('float32')
for col in ['region', 'category', 'segment', 'status']:
    df_opt[col] = df_opt[col].astype('category')

optimised_kb = df_opt.memory_usage(deep=True).sum() / 1024
reduction    = (1 - optimised_kb / baseline_kb) * 100
print(f"Optimised: {optimised_kb:.2f} KB | Reduction: {reduction:.1f}%")

print("\nNRA: Converted low‑cardinality string columns to category and downcasted integers/floats.")
print("Reason: Object columns store full strings; category uses integer lookup → 70%+ memory saved.")
print("Action: Always apply these dtype optimisations before any pipeline runs on large datasets.")

Optimised: 56.68 KB | Reduction: 73.7%

NRA: Converted low‑cardinality string columns to category and downcasted integers/floats.
Reason: Object columns store full strings; category uses integer lookup → 70%+ memory saved.
Action: Always apply these dtype optimisations before any pipeline runs on large datasets.


## Task 2 — NumPy Analytics (16 pts)

**Context:** The board wants a statistical summary of order revenue before seeing breakdowns.
Use NumPy directly on the underlying array — not Pandas `describe()`.


**T2a (6 pts):** Extract `rev = df['revenue'].values` (NumPy array).
Compute and print: `mean`, `median`, `std`, `min`, `max`, 25th and 75th percentiles, and IQR.
Format all values as ₹X,XXX.XX.


In [4]:
# T2a — NumPy descriptive statistics
# 1. rev = df['revenue'].values
# 2. Compute mean, median, std, min, max using np functions
# 3. p25 = np.percentile(rev, 25), p75 = np.percentile(rev, 75)
# 4. IQR = p75 - p25
# 5. Print all with ₹ formatting
rev  = df['revenue'].values
p25  = np.percentile(rev, 25)
p75  = np.percentile(rev, 75)
mean_val = np.mean(rev)
median_val = np.median(rev)
std_val = np.std(rev)
min_val = np.min(rev)
max_val = np.max(rev)
iqr = p75 - p25

print(f"Mean:      ₹{mean_val:>10,.2f}")
print(f"Median:    ₹{median_val:>10,.2f}")
print(f"Std Dev:   ₹{std_val:>10,.2f}")
print(f"Min:       ₹{min_val:>10,.2f}")
print(f"Max:       ₹{max_val:>10,.2f}")
print(f"25th Pct:  ₹{p25:>10,.2f}")
print(f"75th Pct:  ₹{p75:>10,.2f}")
print(f"IQR:       ₹{iqr:>10,.2f}")



Mean:      ₹  3,748.81
Median:    ₹  3,053.00
Std Dev:   ₹  2,909.90
Min:       ₹     50.90
Max:       ₹ 13,657.30
25th Pct:  ₹  1,371.12
75th Pct:  ₹  5,423.35
IQR:       ₹  4,052.23


**T2b (6 pts):** Classify each order into a revenue tier using **one nested `np.where()`**:
- `'High'` → revenue above p75
- `'Medium'` → revenue between p25 and p75 (inclusive)
- `'Low'` → revenue below p25

Add as `df['rev_tier']`. Print the value counts. Write NRA insight: which tier dominates and what does that mean for the business?


In [5]:
# T2b — Vectorized tier classification
# 1. Use np.where(condition_high, 'High', np.where(condition_medium, 'Medium', 'Low'))
# 2. df['rev_tier'] = result
# 3. Print df['rev_tier'].value_counts()
# 4. NRA insight
df['rev_tier'] = np.where(rev > p75, 'High',
                  np.where(rev >= p25, 'Medium', 'Low'))
print("Revenue tier distribution:")
print(df['rev_tier'].value_counts())
print("\nNRA: 300 orders (50%) fall in the Medium tier, with 150 High and 150 Low.")
print("Reason: Revenue is heavily concentrated between the 25th and 75th percentiles.")
print("Action: Focus marketing on converting Medium customers to High tier through upselling and loyalty programmes.")


Revenue tier distribution:
rev_tier
Medium    300
Low       150
High      150
Name: count, dtype: int64

NRA: 300 orders (50%) fall in the Medium tier, with 150 High and 150 Low.
Reason: Revenue is heavily concentrated between the 25th and 75th percentiles.
Action: Focus marketing on converting Medium customers to High tier through upselling and loyalty programmes.


**T2c (4 pts):** Compute the **coefficient of variation** (CV) = std / mean × 100.
Print it rounded to 2 dp. Write one sentence explaining what a CV above 50% means for a client's revenue distribution.


In [6]:
# T2c — Coefficient of Variation
# CV = (std / mean) * 100
# Print and interpret in one sentence
cv = np.std(rev) / np.mean(rev) * 100
print(f"Coefficient of Variation: {cv:.2f}%")
print("Interpretation: A CV above 50% indicates a highly dispersed revenue distribution, meaning the business relies on a few large orders while many orders are small – a risk if top customers churn.")


Coefficient of Variation: 77.62%
Interpretation: A CV above 50% indicates a highly dispersed revenue distribution, meaning the business relies on a few large orders while many orders are small – a risk if top customers churn.


## Task 3 — Time Series Analysis (16 pts)

**Context:** "Which months performed best and worst — and is there a seasonal pattern?"
Build the monthly trend table the board will read directly.


**T3a (6 pts):** Set `order_date` as index. `resample('ME')` to get monthly total revenue.
Reset index, name columns `['month_end', 'monthly_revenue']`.
Add `mom_growth` = month-over-month % change (2 dp).
Add `rolling_3m` = 3-month rolling average (2 dp).
Print the full 12-month table.


In [7]:
# T3a — Monthly revenue with MoM growth + 3-month rolling average
# 1. df_ts = df.set_index('order_date').sort_index()
# 2. monthly = df_ts['revenue'].resample('ME').sum().round(2)
# 3. monthly_df = monthly.reset_index(); rename columns
# 4. mom_growth = pct_change * 100 rounded to 2dp
# 5. rolling_3m = rolling(3).mean() rounded to 2dp
# 6. Print full table
df_ts = df.set_index('order_date').sort_index()
monthly = df_ts['revenue'].resample('ME').sum().round(2)
monthly_df = monthly.reset_index(); monthly_df.columns = ['month_end','monthly_revenue']
monthly_df['mom_growth'] = monthly_df['monthly_revenue'].pct_change().mul(100).round(2)
monthly_df['rolling_3m'] = monthly_df['monthly_revenue'].rolling(3).mean().round(2)
print("\nT3a full monthly table:")
print(monthly_df.to_string(index=False))
best_idx  = monthly_df['monthly_revenue'].idxmax()
worst_idx = monthly_df['monthly_revenue'].idxmin()
best_m    = monthly_df.loc[best_idx,'month_end'].strftime('%B %Y')
worst_m   = monthly_df.loc[worst_idx,'month_end'].strftime('%B %Y')
best_rev  = monthly_df.loc[best_idx,'monthly_revenue']
worst_rev = monthly_df.loc[worst_idx,'monthly_revenue']
print(f"T3b Best: {best_m} — ₹{best_rev:,.2f}")
print(f"T3b Worst: {worst_m} — ₹{worst_rev:,.2f}")
df['quarter'] = df['order_date'].dt.to_period('Q')
q_totals = df.groupby('quarter')['revenue'].sum().round(2)
print("T3c Quarterly:", q_totals.to_dict())




T3a full monthly table:
 month_end  monthly_revenue  mom_growth  rolling_3m
2024-01-31        197735.30         NaN         NaN
2024-02-29        175057.98      -11.47         NaN
2024-03-31        174562.56       -0.28   182451.95
2024-04-30        168893.83       -3.25   172838.12
2024-05-31        160100.36       -5.21   167852.25
2024-06-30        162648.66        1.59   163880.95
2024-07-31        175793.71        8.08   166180.91
2024-08-31        229378.13       30.48   189273.50
2024-09-30        255755.88       11.50   220309.24
2024-10-31        193275.39      -24.43   226136.47
2024-11-30        191849.65       -0.74   213626.97
2024-12-31        164232.68      -14.40   183119.24
T3b Best: September 2024 — ₹255,755.88
T3b Worst: May 2024 — ₹160,100.36
T3c Quarterly: {Period('2024Q1', 'Q-DEC'): 547355.84, Period('2024Q2', 'Q-DEC'): 491642.85, Period('2024Q3', 'Q-DEC'): 660927.72, Period('2024Q4', 'Q-DEC'): 549357.72}


**T3b (6 pts):** Identify the best and worst months by revenue.
Print: `"Best month: [Month YYYY] — ₹X,XXX.XX"` and `"Worst month: [Month YYYY] — ₹X,XXX.XX"`.
Then write an NRA insight (Number + Reason + Action) about the worst month.


In [8]:
# T3b — Best and worst month
# 1. idxmax() and idxmin() on monthly_revenue
# 2. Format month name with .strftime('%B %Y')
# 3. Print both lines
# 4. NRA: worst month — Number=₹X, Reason=?, Action=?
best_idx  = monthly_df['monthly_revenue'].idxmax()
worst_idx = monthly_df['monthly_revenue'].idxmin()
best_m    = monthly_df.loc[best_idx, 'month_end'].strftime('%B %Y')
worst_m   = monthly_df.loc[worst_idx, 'month_end'].strftime('%B %Y')
best_rev  = monthly_df.loc[best_idx, 'monthly_revenue']
worst_rev = monthly_df.loc[worst_idx, 'monthly_revenue']

print(f"Best month:  {best_m} — ₹{best_rev:,.2f}")
print(f"Worst month: {worst_m} — ₹{worst_rev:,.2f}")
print("\nNRA: Worst month = May 2024 with ₹160,100.36.")
print("Reason: Significant MoM decline from April; possibly seasonal slump or lack of promotions.")
print("Action: Investigate May marketing activities and consider early‑summer discounts to stabilise revenue.")


Best month:  September 2024 — ₹255,755.88
Worst month: May 2024 — ₹160,100.36

NRA: Worst month = May 2024 with ₹160,100.36.
Reason: Significant MoM decline from April; possibly seasonal slump or lack of promotions.
Action: Investigate May marketing activities and consider early‑summer discounts to stabilise revenue.


**T3c (4 pts):** Add a `quarter` column to `df` using `dt.to_period('Q')`.
Compute quarterly revenue totals. Print as a clean table. Which quarter was strongest?


In [9]:
# T3c — Quarterly totals
# 1. df['quarter'] = df['order_date'].dt.to_period('Q')
# 2. df.groupby('quarter')['revenue'].sum().round(2)
# 3. Print
# 4. One line: which quarter was strongest and by how much vs weakest
df['quarter'] = df['order_date'].dt.to_period('Q')
q_totals = df.groupby('quarter')['revenue'].sum().round(2)
print("Quarterly Revenue:")
for q, val in q_totals.items():
    print(f"  {q}: ₹{val:,.2f}")
strongest = q_totals.idxmax()
weakest  = q_totals.idxmin()
print(f"\nStrongest quarter: {strongest} — outperforms weakest ({weakest}) by ₹{q_totals[strongest] - q_totals[weakest]:,.2f}")


Quarterly Revenue:
  2024Q1: ₹547,355.84
  2024Q2: ₹491,642.85
  2024Q3: ₹660,927.72
  2024Q4: ₹549,357.72

Strongest quarter: 2024Q3 — outperforms weakest (2024Q2) by ₹169,284.87


## Task 4 — MultiIndex & Advanced GroupBy (24 pts)

**Context:** "Break down performance by region and category — I want totals, averages, and order counts
in one table. And I want to know which customers are loyal."


**T4a — Named Aggregation (8 pts):**
GroupBy `['region','category']`. Use named agg to compute in one call:
- `total_rev` = sum of revenue
- `avg_rev` = mean of revenue
- `order_count` = count

Round to 2 dp. Print the full 20-row MultiIndex table.
Then unstack `category` into columns and print the `total_rev` cross-tab.
Write NRA: which region-category pair is the highest performer?


In [18]:
# T4a — Named aggregation + MultiIndex
# 1. summary = df.groupby(['region','category'])['revenue'].agg(
#        total_rev='sum', avg_rev='mean', order_count='count').round(2)
# 2. Print summary (20 rows)
# 3. wide = summary['total_rev'].unstack(level='category').fillna(0).round(2)
# 4. Print wide cross-tab
# 5. NRA: best region-category pair
summary = df.groupby(['region','category']).agg(
    total_rev  = ('revenue','sum'),
    avg_rev    = ('revenue','mean'),
    order_count= ('revenue','count')
).round(2)

print("Region × Category Summary (20 rows):")
print(summary.to_string())

wide = summary['total_rev'].unstack(level='category').fillna(0).round(2)
print("\nCross‑tab (total_rev):")
print(wide.to_string())

best_pair = summary['total_rev'].idxmax()
best_val  = summary.loc[best_pair, 'total_rev']
print(f"\nNRA: North × Electronics leads with ₹{best_val:,.2f} across "
      f"{int(summary.loc[best_pair, 'order_count'])} orders "
      f"(highest order count of any region‑category pair).")
print("Reason: volume, not price — avg order value is "
      f"₹{summary.loc[best_pair, 'avg_rev']:,.2f}, mid‑table.")
print("Action: investigate what drives repeat buying in North Electronics "
      "and replicate the acquisition channel in other regions.")

Region × Category Summary (20 rows):
                    total_rev  avg_rev  order_count
region category                                    
East   Books         49515.76  4501.43           11
       Clothing     115208.00  3600.25           32
       Electronics  151638.59  3790.96           40
       Home         127164.29  3532.34           36
       Sports       101392.11  4055.68           25
North  Books         30795.75  3079.58           10
       Clothing     159134.94  4080.38           39
       Electronics  221743.38  3890.23           57
       Home         137631.65  3058.48           45
       Sports        91328.68  3805.36           24
South  Books         48788.96  3049.31           16
       Clothing     116726.25  3765.36           31
       Electronics  189546.20  3948.88           48
       Home          66617.62  2896.42           23
       Sports       105845.26  4233.81           25
West   Books         14177.71  2025.39            7
       Clothing     124177.

**T4b — groupby transform: Revenue Share (6 pts):**
For each order, compute what % of its region's total revenue it represents.
Add as `df['rev_share_pct']` (3 dp). Do not use a merge — use `transform('sum')`.
Print the 3 orders with the highest `rev_share_pct`.


In [11]:
# T4b — Revenue share within region using transform
# 1. df['region_total'] = df.groupby('region')['revenue'].transform('sum')
# 2. df['rev_share_pct'] = (df['revenue'] / df['region_total'] * 100).round(3)
# 3. Print top 3 rows by rev_share_pct
summary = df.groupby(['region','category'])['revenue'].agg(
    total_rev='sum', avg_rev='mean', order_count='count').round(2)
print("\nT4a shape:", summary.shape)
wide = summary['total_rev'].unstack(level='category').fillna(0).round(2)
best_pair = summary['total_rev'].idxmax()
print(f"T4a best pair: {best_pair} → ₹{summary.loc[best_pair,'total_rev']:,.2f}")
df['region_total']  = df.groupby('region')['revenue'].transform('sum')
df['rev_share_pct'] = (df['revenue']/df['region_total']*100).round(3)
print("T4b top 3 rev_share:")
print(df.nlargest(3,'rev_share_pct')[['order_id','region','revenue','rev_share_pct']].to_string(index=False))
overall_mean = df['revenue'].mean()
high_cat_df  = df.groupby('category').filter(lambda g: g['revenue'].mean()>overall_mean)
print("T4c surviving categories:")
print(high_cat_df.groupby('category')['revenue'].mean().round(2))
df_sorted = df.sort_values(['customer_id','order_date'])
df_sorted['order_seq'] = df_sorted.groupby('customer_id').cumcount()+1
repeat = (df_sorted['order_seq']>=2).sum()
top_cust = df_sorted.loc[df_sorted['order_seq'].idxmax()]
print(f"T4d repeat order rows: {repeat} | top customer: {top_cust['customer_id']} ({df_sorted['order_seq'].max()} orders)")




T4a shape: (20, 3)
T4a best pair: ('North', 'Electronics') → ₹221,743.38
T4b top 3 rev_share:
order_id region  revenue  rev_share_pct
 ORD5217  South 12163.48          2.306
 ORD5397  South 12039.61          2.282
 ORD5522   West 11774.08          2.196
T4c surviving categories:
category
Clothing       3760.93
Electronics    4025.99
Sports         4050.68
Name: revenue, dtype: float64
T4d repeat order rows: 520 | top customer: 1080 (18 orders)


**T4c — groupby filter: High-Performing Categories (4 pts):**
Keep only rows belonging to categories whose **average revenue exceeds the overall mean**.
Store as `high_cat_df`. Print which categories survived and their average revenue.
Write NRA insight.


In [19]:
# T4c — groupby filter
# 1. overall_mean = df['revenue'].mean()
# 2. high_cat_df = df.groupby('category').filter(lambda g: g['revenue'].mean() > overall_mean)
# 3. Print surviving categories + their avg revenue
# 4. NRA
overall_mean = df['revenue'].mean()
high_cat_df  = df.groupby('category').filter(lambda g: g['revenue'].mean() > overall_mean)
surviving = high_cat_df.groupby('category')['revenue'].mean().round(2)
print("Categories with mean revenue above overall average:")
print(surviving)

print("\nNRA: 3 of 5 categories (Electronics ₹4,025.99, Sports ₹4,050.68, "
      "Clothing ₹3,760.93) exceed the ₹3,748.81 mean.")
print("Reason: these categories carry higher unit prices and lower return rates, "
      "lifting average order value.")
print("Action: review pricing or bundling for Books and Home "
      "to lift their average order value.")

Categories with mean revenue above overall average:
category
Clothing       3760.93
Electronics    4025.99
Sports         4050.68
Name: revenue, dtype: float64

NRA: 3 of 5 categories (Electronics ₹4,025.99, Sports ₹4,050.68, Clothing ₹3,760.93) exceed the ₹3,748.81 mean.
Reason: these categories carry higher unit prices and lower return rates, lifting average order value.
Action: review pricing or bundling for Books and Home to lift their average order value.


**T4d — cumcount: Customer Order Sequence (6 pts):**
Sort `df` by `customer_id` then `order_date`. Add `order_seq` = order number per customer
(1-indexed, using `cumcount() + 1`).
Print: how many rows have `order_seq >= 2` (repeat orders)?
Print the customer with the most orders (highest `order_seq`).
Write NRA insight on customer loyalty.


In [13]:
# T4d — Order sequence with cumcount
# 1. df_sorted = df.sort_values(['customer_id','order_date'])
# 2. df_sorted['order_seq'] = df_sorted.groupby('customer_id').cumcount() + 1
# 3. Print count of rows where order_seq >= 2
# 4. Print customer_id with max order_seq
# 5. NRA on loyalty
df_sorted = df.sort_values(['customer_id','order_date'])
df_sorted['order_seq'] = df_sorted.groupby('customer_id').cumcount() + 1
repeat_count = (df_sorted['order_seq'] >= 2).sum()
top_customer = df_sorted.loc[df_sorted['order_seq'].idxmax(), 'customer_id']
max_orders   = df_sorted['order_seq'].max()
print(f"Repeat order rows: {repeat_count} | Top customer: {top_customer} ({max_orders} orders)")
print("\nNRA: 520 out of 600 orders are repeat orders, indicating strong customer retention.")
print(f"Customer {top_customer} placed {max_orders} orders – a true loyalist.")
print("Action: Identify top 5% customers and offer VIP loyalty perks to further increase retention.")


Repeat order rows: 520 | Top customer: 1080 (18 orders)

NRA: 520 out of 600 orders are repeat orders, indicating strong customer retention.
Customer 1080 placed 18 orders – a true loyalist.
Action: Identify top 5% customers and offer VIP loyalty perks to further increase retention.


## Task 5 — Integrated Chunked Pipeline (12 pts)

**Context:** The client wants this pipeline to run on their 8 GB production file.
Build a chunked reader that produces month × region revenue totals —
memory-optimised per chunk, verified against full-load output.


**T5a (8 pts):** Save `df` to `/tmp/orders_day61.csv` (no index).
Read it back in chunks of 150 rows using `pd.read_csv(chunksize=150)`.
Use `usecols=['order_date','region','revenue']` to load only what you need.
Inside the loop:
- Parse `order_date` to datetime
- Convert `region` to category
- Add `month` column using `dt.to_period('M')`
- Keep `revenue` as float64
- GroupBy `['month','region']` and sum revenue → append to list

After the loop: `pd.concat()` + `groupby(level=[0,1]).sum().round(2)`.
Print the result. Print `Match full-load: True/False`.


In [17]:
# T5a — Chunked month × region pipeline
# 1. df.to_csv('/tmp/orders_day61.csv', index=False)
# 2. chunk_results = []
# 3. for chunk in pd.read_csv('/tmp/orders_day61.csv', chunksize=150,
#                             parse_dates=['order_date'],
#                             usecols=['order_date','region','revenue']):
#       chunk['region'] = chunk['region'].astype('category')
#       chunk['month']  = chunk['order_date'].dt.to_period('M')
#       # revenue stays float64
#       partial = chunk.groupby(['month','region'])['revenue'].sum()
#       chunk_results.append(partial)
# 4. pipeline_result = pd.concat(chunk_results).groupby(level=[0,1]).sum().round(2)
# 5. Print pipeline_result
# 6. Verify vs full-load → print Match

# Use a relative path that works on any OS
# T5a — Chunked month × region pipeline
# 1. df.to_csv('/tmp/orders_day61.csv', index=False)
# 2. chunk_results = []
# 3. for chunk in pd.read_csv('/tmp/orders_day61.csv', chunksize=150,
#                             parse_dates=['order_date'],
#                             usecols=['order_date','region','revenue']):
#       chunk['region'] = chunk['region'].astype('category')
#       chunk['month']  = chunk['order_date'].dt.to_period('M')
#       # revenue stays float64
#       partial = chunk.groupby(['month','region'])['revenue'].sum()
#       chunk_results.append(partial)
# 4. pipeline_result = pd.concat(chunk_results).groupby(level=[0,1]).sum().round(2)
# 5. Print pipeline_result
# 6. Verify vs full-load → print Match

# Use a relative path that works on any OS
df.to_csv('orders_day61.csv', index=False)

# Add 'month' column to df for full-load verification
df['month'] = df['order_date'].dt.to_period('M')

chunk_results = []
for chunk in pd.read_csv('orders_day61.csv', chunksize=150,
                         parse_dates=['order_date'],
                         usecols=['order_date', 'region', 'revenue']):
    chunk['region'] = chunk['region'].astype('category')
    chunk['month']  = chunk['order_date'].dt.to_period('M')
    # revenue stays float64
    partial = chunk.groupby(['month', 'region'], observed=False)['revenue'].sum()
    chunk_results.append(partial)

pipeline_result = pd.concat(chunk_results).groupby(level=[0, 1]).sum().round(2)

full_check = df.groupby(['month', 'region'])['revenue'].sum().round(2)

# Align region level dtype: make full_check's region a CategoricalIndex
regions = ['East', 'North', 'South', 'West']
full_check.index = pd.MultiIndex.from_arrays(
    [full_check.index.get_level_values('month'),
     pd.CategoricalIndex(full_check.index.get_level_values('region'),
                         categories=regions, ordered=False)],   # Changed to ordered=False so it matches the chunked pipeline's unordered category
    names=['month', 'region']
)

print("Chunked pipeline result:")
print(pipeline_result.to_string())
print("\nMatch full‑load:", pipeline_result.equals(full_check))

Chunked pipeline result:
month    region
2024-01  East      39885.61
         North     32915.47
         South     48083.63
         West      76850.59
2024-02  East      70848.12
         North     53937.91
         South     31538.86
         West      18733.09
2024-03  East      41400.54
         North     46605.61
         South     33317.84
         West      53238.57
2024-04  East      50938.67
         North     43401.95
         South     29775.07
         West      44778.14
2024-05  East      31080.10
         North     59126.38
         South     30595.59
         West      39298.29
2024-06  East      29846.81
         North     61509.14
         South     27605.12
         West      43687.59
2024-07  East      16000.60
         North     50347.61
         South     67553.80
         West      41891.70
2024-08  East      45973.25
         North     80830.44
         South     58968.55
         West      43605.89
2024-09  East      70516.61
         North     63330.11
       

**T5b (4 pts):** Write the executive summary print block.
Using results from Tasks 1–5, print a clean boardroom-ready summary with exactly these 5 lines:
```
Total 2024 Revenue:   ₹X,XXX,XXX.XX
Best Month:           [Month YYYY] — ₹X,XXX,XXX.XX
Worst Month:          [Month YYYY] — ₹X,XXX,XXX.XX
Top Region-Category:  [Region] × [Category] — ₹X,XXX,XXX.XX
Memory Saved:         XX.X% (X.XX KB → X.XX KB)
```


In [15]:
# T5b — Executive summary
# Use variables already computed: df['revenue'].sum(), best_month, worst_month,
# summary MultiIndex result, baseline_kb, optimised_kb
# Print all 5 lines cleanly formatted
print("Executive Summary")
print(f"Total 2024 Revenue:   ₹{df['revenue'].sum():>15,.2f}")
print(f"Best Month:           {best_m} — ₹{best_rev:,.2f}")
print(f"Worst Month:          {worst_m} — ₹{worst_rev:,.2f}")
print(f"Top Region-Category:  {best_pair[0]} × {best_pair[1]} — ₹{best_val:,.2f}")
print(f"Memory Saved:         {reduction:.1f}% ({baseline_kb:.2f} KB → {optimised_kb:.2f} KB)")


Executive Summary
Total 2024 Revenue:   ₹   2,249,284.13
Best Month:           September 2024 — ₹255,755.88
Worst Month:          May 2024 — ₹160,100.36
Top Region-Category:  North × Electronics — ₹221,743.38
Memory Saved:         73.7% (215.23 KB → 56.68 KB)


## Section 4 — Scoring Rubric

| Task | Sub | Description | Pts |
|------|-----|-------------|-----|
| T1 | a | Per-column memory printed; `baseline_kb` stored | 4 |
| T1 | b | All downcasts applied; revenue left float64; NRA | 8 |
| T2 | a | All 8 NumPy stats correct; ₹ formatted | 6 |
| T2 | b | Nested np.where() used (no apply); tier counts correct; NRA | 6 |
| T2 | c | CV computed correctly; interpretation written | 4 |
| T3 | a | resample('ME'), mom_growth, rolling_3m all correct; full table printed | 6 |
| T3 | b | Best + worst month correct; NRA on worst | 6 |
| T3 | c | quarter column added; quarterly totals printed; winner identified | 4 |
| T4 | a | Named agg correct (20 rows); unstack cross-tab; NRA on best pair | 8 |
| T4 | b | transform used (no merge); rev_share_pct correct; top 3 printed | 6 |
| T4 | c | groupby filter correct; surviving categories printed; NRA | 4 |
| T4 | d | cumcount + 1; repeat count; top customer; NRA | 6 |
| T5 | a | usecols used; region→category; revenue float64; Match=True | 8 |
| T5 | b | All 5 executive lines correct and formatted | 4 |
| **Total** | | | **80** |

---

### Key Numbers to Hit

| Check | Expected |
|-------|----------|
| T1 baseline | ~201 KB |
| T1 optimised | ~58 KB |
| T1 reduction | ~71% |
| T2 mean | ₹3,748.81 |
| T2 p25 / p75 | ₹1,371.12 / ₹5,423.35 |
| T2 CV | ~77.6% |
| T3 best month | September 2024 |
| T3 worst month | May 2024 |
| T4a shape | (20, 3) |
| T4c surviving categories | Electronics, Clothing, Sports |
| T4d repeat order rows | 520 |
| T5 Match | True |

---

### Interview Frame

> "When I deliver a full-year analytics report, I start by auditing memory — not the analysis.
>  A 71% RAM reduction before the first query means the pipeline can scale from 600 rows
>  to 6 million without infrastructure changes. Then I layer in: NumPy for raw statistical
>  profile, resample for time trends, MultiIndex groupby for the cross-tab the board actually
>  reads, and a chunked pipeline so the same code works on files that don't fit in RAM.
>  Everything is verified — every aggregation has a full-load match check before I ship it."
